In [13]:
!pip install yt-dlp

^C


In [15]:
from yt_dlp import YoutubeDL

url = "https://www.youtube.com/watch?v=ajzQj7bjSWE"

opciones = {
    "format": "bestvideo[height<=720]+bestaudio/best[height<=720]",
    "outtmpl": "videos_descargados/%(title)s.%(ext)s",
    "cookiefile": "/content/cookies_casa.txt",   # << IMPORTANTE
}

with YoutubeDL(opciones) as ydl:
    ydl.download([url])


[youtube] Extracting URL: https://www.youtube.com/watch?v=ajzQj7bjSWE
[youtube] ajzQj7bjSWE: Downloading webpage


[youtube] ajzQj7bjSWE: Downloading android sdkless player API JSON
[youtube] ajzQj7bjSWE: Downloading web safari player API JSON


[youtube] ajzQj7bjSWE: Downloading m3u8 information


[info] ajzQj7bjSWE: Downloading 1 format(s): 398+251-3


ERROR: You have requested merging of multiple formats but ffmpeg is not installed. Aborting due to --abort-on-error


DownloadError: ERROR: You have requested merging of multiple formats but ffmpeg is not installed. Aborting due to --abort-on-error

# Degradar vídeo en buena calidad para conseguir el pair para entrenar autoencoder

In [4]:
import cv2
import numpy as np
import os

In [21]:
videos_descargados = "C:/Users/javie/Documents/Workshop3/videos_descargados"

In [26]:
# Input paths

video1_path = "videos_descargados/Race Highlights ｜ 2025 Monaco Grand Prix.mp4"   # high-quality modern video in 720p
video2_path = "videos_descargados/Race Highlights ｜ 2025 Monaco Grand Prix_lowQ.mp4"  # degraded output video (done it below)



In [14]:

# Open the original video
cap = cv2.VideoCapture(video1_path)

# Video properties
fps = cap.get(cv2.CAP_PROP_FPS)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

print(f"The video properties of the original video are: {fps} fps, {width} width and {height} height")

# Output video writer (mp4)
fourcc = cv2.VideoWriter_fourcc(*"mp4v")
out = cv2.VideoWriter(video2_path, fourcc, fps, (width, height))

# Function to degrade frames of the original video
def degrade_frame(frame):
    # Convert to RGB for manipulation
    frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    # 1) Downscale and upscale (simulates old camera / low resolution)
    down_factor = 3  # higher = more pixelation # adds pixelation
    small = cv2.resize(frame, (width // down_factor, height // down_factor),
                       interpolation=cv2.INTER_AREA)
    degraded = cv2.resize(small, (width, height), interpolation=cv2.INTER_LINEAR)

    # 2) Add blur (simulates old optics or bad transmission)
    degraded = cv2.GaussianBlur(degraded, (7, 7), 0)

    # 3) Desaturate (old videos had weaker colors)
    hsv = cv2.cvtColor(degraded, cv2.COLOR_RGB2HSV)
    hsv[..., 1] = hsv[..., 1] * 0.4  # reduce saturation to 40%
    degraded = cv2.cvtColor(hsv, cv2.COLOR_HSV2RGB)

    # 4) Add noise (simulates analog interference)
    noise = np.random.normal(0, 12, degraded.shape).astype(np.float32)
    degraded = degraded.astype(np.float32) + noise
    degraded = np.clip(degraded, 0, 255).astype(np.uint8)

    # 5) Simulate old JPEG compression artifacts
    encode_param = [int(cv2.IMWRITE_JPEG_QUALITY), 40]  # very low quality
    _, encimg = cv2.imencode('.jpg', cv2.cvtColor(degraded, cv2.COLOR_RGB2BGR), encode_param)
    degraded = cv2.imdecode(encimg, cv2.IMREAD_COLOR)

    return degraded

# -----------------------------
# Process the whole video
# -----------------------------
frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
print("Total frames:", frame_count)

i = 0
while True:
    ret, frame = cap.read()
    if not ret:
        break

    degraded_frame = degrade_frame(frame)
    out.write(degraded_frame)

    if i % 100 == 0:
        print(f"Processing frame {i}/{frame_count}")
    i += 1

cap.release()
out.release()

print("Degraded video saved at:", video2_path)


The video properties of the original video are: 0.0 fps, 0 width and 0 height
Total frames: 0
Degraded video saved at: /content/videos_descargados/Race Highlights ｜ 2025 Monaco Grand Prix_lowQ.mp4


   ---------------------------------------- 0.0/3.3 MB ? eta -:--:--
   --- ------------------------------------ 0.3/3.3 MB ? eta -:--:--
   ------ --------------------------------- 0.5/3.3 MB 1.3 MB/s eta 0:00:03
   --------- ------------------------------ 0.8/3.3 MB 1.3 MB/s eta 0:00:02
   ------------ --------------------------- 1.0/3.3 MB 1.4 MB/s eta 0:00:02
   --------------- ------------------------ 1.3/3.3 MB 1.4 MB/s eta 0:00:02
   ------------------- -------------------- 1.6/3.3 MB 1.4 MB/s eta 0:00:02
   ------------------------- -------------- 2.1/3.3 MB 1.5 MB/s eta 0:00:01
   ---------------------------- ----------- 2.4/3.3 MB 1.5 MB/s eta 0:00:01
   ------------------------------- -------- 2.6/3.3 MB 1.5 MB/s eta 0:00:01
   -------------------------------------- - 3.1/3.3 MB 1.6 MB/s eta 0:00:01
   -------------------------------------- - 3.1/3.3 MB 1.6 MB/s eta 0:00:01
   ---------------------------------------- 3.3/3.3 MB 1.3 MB/s  0:00:02


In [33]:
# Create the folders
import os
output_folder1 = "/content/frames_720p"
output_folder2 = "/content/frames_lowQ"

os.makedirs(output_folder1, exist_ok=True)
os.makedirs(output_folder2, exist_ok=True)

# Read the videos
cap1 = cv2.VideoCapture(video1_path)
cap2 = cv2.VideoCapture(video2_path)

# Get all frames (CORRECTED)
total_frames1 = int(cap1.get(cv2.CAP_PROP_FRAME_COUNT))
total_frames2 = int(cap2.get(cv2.CAP_PROP_FRAME_COUNT))

# Print the total frames
print("Total frames 1 (HQ):", total_frames1)
print("Total frames 2 (LQ):", total_frames2)

Total frames 1 (HQ): 11164
Total frames 2 (LQ): 11164


In [1]:
# Preprocessing
import cv2
import os

# Extract frames
for i in range(total_frames1):
  cap1.set(cv2.CAP_PROP_POS_FRAMES, i)
  ret, frame = cap1.read()
  if ret:
    cv2.imwrite(os.path.join(output_folder1, f"frame_{i}.jpg"), frame)
# Uniform sampling from the video with more frames (lowQ)
step_size = total_frame1 / total_frame2
for i in range(total_frame2):
    frame_pos = int(i*step_size)
    cap2.set(cv2.CAP_PROP_POS_FRAMES, frame_pos)
    ret, frame = cap2.read()
    if ret:
      cv2.imwrite(os.path.join(output_folder2, f"frame_{i}.jpg"), frame)

# Release
cap1.release()
cap2.release()
cv2.destroyAllWindows()

NameError: name 'total_frames1' is not defined

In [29]:
import cv2
import os

# ----- FAST EXTRACTION OF HQ FRAMES -----

cap1 = cv2.VideoCapture(video1_path)
i = 0

while True:
    ret, frame = cap1.read()
    if not ret:
        break
    cv2.imwrite(os.path.join(output_folder1, f"frame_{i:05d}.jpg"), frame)
    i += 1

cap1.release()

# ----- FAST EXTRACTION OF LQ FRAMES WITH UNIFORM SAMPLING -----

cap2 = cv2.VideoCapture(video2_path)

# Recompute frame counts in case they were wrong
total_frames1 = len(os.listdir(output_folder1))
total_frames2 = int(cap2.get(cv2.CAP_PROP_FRAME_COUNT))

step_size = total_frames2 / total_frames1

frame_positions = [int(i * step_size) for i in range(total_frames1)]

i = 0
current_pos = 0

for pos in frame_positions:
    # Fast forward until reaching the target frame
    while current_pos < pos:
        ret = cap2.grab()  # MUCH FASTER than reading and discarding
        current_pos += 1

    ret, frame_lq = cap2.retrieve()  # Read the selected frame
    if ret:
        cv2.imwrite(os.path.join(output_folder2, f"frame_{i:05d}.jpg"), frame_lq)
    i += 1

cap2.release()
cv2.destroyAllWindows()


In [30]:
# PAth to low quality frame
frame_path = "/content/frames_lowQ/frame_01092.jpg"
frame = cv2.imread(frame_path)
height, width, channels = frame.shape
print(f"Frame 1092 size: {height}x{width}x{channels}")

# Pah to high quality frame
frame_path = "/content/frames_720p/frame_01092.jpg"
frame = cv2.imread(frame_path)
height, width, channels = frame.shape
print(f"Frame 1092 size: {height}x{width}x{channels}")

Frame 1092 size: 720x1280x3
Frame 1092 size: 720x1280x3


In [ ]:
import tensorflow as tf
import numpy as np
import cv2
import os

def create_dataset(lowq_dir, highq_dir, batch_size):
    """
    Creates a TensorFlow dataset that dynamically loads image pairs.

    Args:
        lowq_dir (str): directory containing low-quality frames
        highq_dir (str): directory containing high-quality frames
        batch_size (int)

    Returns:
        dataset (tf.data.Dataset)
        steps_per_epoch (int)
    """

    def load_image_pair(lowq_filepath, highq_filepath):
        """Loads and normalizes one pair of images."""
        lowq_image = cv2.imread(lowq_filepath)
        highq_image = cv2.imread(highq_filepath)

        lowq_image = lowq_image.astype(np.float32) / 255.0
        highq_image = highq_image.astype(np.float32) / 255.0

        return lowq_image, highq_image

    # Get sorted lists of filepaths
    lowq_filepaths_list = sorted(os.listdir(lowq_dir))
    highq_filepaths_list = sorted(os.listdir(highq_dir))

    assert len(lowq_filepaths_list) == len(highq_filepaths_list), "Frame count mismatch"

    # Convert filenames → full paths
    lowq_filepaths_list = [os.path.join(lowq_dir, f) for f in lowq_filepaths_list]
    highq_filepaths_list = [os.path.join(highq_dir, f) for f in highq_filepaths_list]

    # Detect image shape from first pair
    lowq_image, highq_image = load_image_pair(lowq_filepaths_list[0], highq_filepaths_list[0])
    lowq_shape = lowq_image.shape
    highq_shape = highq_image.shape

    def generator():
        """Yields one (lowq_image, highq_image) pair per iteration."""
        for lq_fp, hq_fp in zip(lowq_filepaths_list, highq_filepaths_list):
            yield load_image_pair(lq_fp, hq_fp)

    # Create dataset
    dataset = tf.data.Dataset.from_generator(
        generator,
        output_signature=(
            tf.TensorSpec(shape=lowq_shape, dtype=tf.float32),
            tf.TensorSpec(shape=highq_shape, dtype=tf.float32)
        )
    )

    dataset = dataset.batch(batch_size).prefetch(tf.data.AUTOTUNE)

    # Steps per epoch
    total_num_samples = len(lowq_filepaths_list)
    steps_per_epoch = total_num_samples // batch_size

    return dataset, steps_per_epoch


In [ ]:
# BUILD THE MODEL

from tensorflow.keras.layers import Input, Conv2D, Conv2DTranspose, Resizing
from tensorflow.keras.models import Model

def build_autoencoder(input_shape, output_shape=(720, 1280, 3)):
    """
    Builds an autoencoder that upsamples low-quality frames to high-quality.
    """

    inputs = Input(shape=input_shape)

    # ------- ENCODER -------
    x = Conv2D(32, 3, strides=2, padding='same', activation='relu')(inputs)
    x = Conv2D(16, 3, strides=2, padding='same', activation='relu')(x)
    x = Conv2D(128, 3, strides=2, padding='same', activation='relu')(x)

    # ------- DECODER -------
    x = Conv2DTranspose(32, 3, strides=2, padding='same', activation='relu')(x)
    x = Conv2DTranspose(32, 3, strides=2, padding='same', activation='relu')(x)
    x = Conv2DTranspose(8, 3, strides=2, padding='same', activation='relu')(x)

    # Resize to final expected resolution (high-quality)
    x = Resizing(output_shape[0], output_shape[1], interpolation='bilinear')(x)

    # Final output layer
    outputs = Conv2D(3, 3, strides=1, padding='same', activation='sigmoid')(x)

    model = Model(inputs, outputs)
    return model


In [31]:
import os

lowq_dir = "/content/frames_lowQ"
highq_dir = "/content/frames_720p"

lowq_frames = sorted(os.listdir(lowq_dir))
highq_frames = sorted(os.listdir(highq_dir))

print("LowQ frames:", len(lowq_frames))
print("HighQ frames:", len(highq_frames))

# optional: inspect first 10 names to check formatting
print("First LowQ:", lowq_frames[:10])
print("First HighQ:", highq_frames[:10])


LowQ frames: 11163
HighQ frames: 11164
First LowQ: ['frame_00001.jpg', 'frame_00002.jpg', 'frame_00003.jpg', 'frame_00004.jpg', 'frame_00005.jpg', 'frame_00006.jpg', 'frame_00007.jpg', 'frame_00008.jpg', 'frame_00009.jpg', 'frame_00010.jpg']
First HighQ: ['frame_00000.jpg', 'frame_00001.jpg', 'frame_00002.jpg', 'frame_00003.jpg', 'frame_00004.jpg', 'frame_00005.jpg', 'frame_00006.jpg', 'frame_00007.jpg', 'frame_00008.jpg', 'frame_00009.jpg']


In [ ]:
import os

highq_dir = "/content/frames_720p"

bad_files = ["frame_0.jpg", "frame_00000.jpg"]

for f in bad_files:
    path = os.path.join(highq_dir, f)
    if os.path.exists(path):
        os.remove(path)
        print("Removed:", path)


In [32]:
lowq_files = sorted(os.listdir(lowq_dir))
highq_files = sorted(os.listdir(highq_dir))

# Find the first matching filename
start_name = lowq_files[0]  # 'frame_00002.jpg'

# Keep only matching highQ frames
highq_files = [f for f in highq_files if f >= start_name]

# Now truncate to same length
min_len = min(len(lowq_files), len(highq_files))

lowq_files = lowq_files[:min_len]
highq_files = highq_files[:min_len]

print("Using frames:", min_len)


Using frames: 11163


In [ ]:
# FIRST CONFIGURATION

lowq_dir = "/content/frames_lowQ"
highq_dir = "/content/frames_720p"

# create dataset
dataset, steps_per_epoch = create_dataset(lowq_dir, highq_dir, batch_size=2)

# get input shape from dataset specification
input_shape = dataset.element_spec[0].shape[1:]  # (H, W, C)

# build model
model = build_autoencoder(input_shape)

# compile model
model.compile(optimizer='adam', loss='mse')

# print summary
model.summary()

# train
model.fit(dataset, epochs=5, steps_per_epoch=steps_per_epoch)


In [ ]:
# Downscale frames before feeding model to speed up training

target_height, target_width = 360, 640
lowq_image = cv2.resize(lowq_image, (target_width, target_height))
highq_image = cv2.resize(highq_image, (target_width, target_height))


In [3]:
import tensorflow as tf
import numpy as np
import cv2
import os

def create_dataset(lowq_dir, highq_dir, batch_size):
    """
    Creates a TensorFlow dataset that dynamically loads image pairs.

    Args:
        lowq_dir (str): directory containing low-quality frames
        highq_dir (str): directory containing high-quality frames
        batch_size (int)

    Returns:
        dataset (tf.data.Dataset)
        steps_per_epoch (int)
    """

    def load_image_pair(lowq_filepath, highq_filepath, target_height=360, target_width=640):
      #Load and resize one pair of frames.
      lowq_image = cv2.imread(lowq_filepath)
      highq_image = cv2.imread(highq_filepath)

      # Resize to smaller resolution
      lowq_image = cv2.resize(lowq_image, (target_width, target_height))
      highq_image = cv2.resize(highq_image, (target_width, target_height))

      # Normalize to 0-1
      lowq_image = lowq_image.astype(np.float32) / 255.0
      highq_image = highq_image.astype(np.float32) / 255.0

      return lowq_image, highq_image

    # Get sorted lists of filepaths
    lowq_filepaths_list = sorted(os.listdir(lowq_dir))
    highq_filepaths_list = sorted(os.listdir(highq_dir))

    assert len(lowq_filepaths_list) == len(highq_filepaths_list), "Frame count mismatch"

    # Convert filenames → full paths
    lowq_filepaths_list = [os.path.join(lowq_dir, f) for f in lowq_filepaths_list]
    highq_filepaths_list = [os.path.join(highq_dir, f) for f in highq_filepaths_list]

    # Detect image shape from first pair
    lowq_image, highq_image = load_image_pair(lowq_filepaths_list[0], highq_filepaths_list[0])
    lowq_shape = lowq_image.shape
    highq_shape = highq_image.shape

    def generator():
        """Yields one (lowq_image, highq_image) pair per iteration."""
        for lq_fp, hq_fp in zip(lowq_filepaths_list, highq_filepaths_list):
          yield load_image_pair(lq_fp, hq_fp, target_height=360, target_width=640)

    # Create dataset
    dataset = tf.data.Dataset.from_generator(
        generator,
        output_signature=(
            tf.TensorSpec(shape=lowq_shape, dtype=tf.float32),
            tf.TensorSpec(shape=highq_shape, dtype=tf.float32)
        )
    )

    dataset = dataset.batch(batch_size).prefetch(tf.data.AUTOTUNE)

    # Steps per epoch
    total_num_samples = len(lowq_filepaths_list)
    steps_per_epoch = total_num_samples // batch_size

    return dataset, steps_per_epoch


ImportError: Traceback (most recent call last):
  File "C:\Users\javie\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tensorflow\python\pywrap_tensorflow.py", line 73, in <module>
    from tensorflow.python._pywrap_tensorflow_internal import *
ImportError: DLL load failed while importing _pywrap_tensorflow_internal: Error en una rutina de inicialización de biblioteca de vínculos dinámicos (DLL).


Failed to load the native TensorFlow runtime.
See https://www.tensorflow.org/install/errors for some common causes and solutions.
If you need help, create an issue at https://github.com/tensorflow/tensorflow/issues and include the entire stack trace above this error message.

In [ ]:
from tensorflow.keras.layers import Input, Conv2D, Conv2DTranspose, Resizing
from tensorflow.keras.models import Model

def build_autoencoder(input_shape, output_shape=(360, 640, 3)):
    inputs = Input(shape=input_shape)

    # ENCODER (smaller)
    x = Conv2D(16, 3, strides=2, padding='same', activation='relu')(inputs)
    x = Conv2D(8, 3, strides=2, padding='same', activation='relu')(x)
    x = Conv2D(64, 3, strides=2, padding='same', activation='relu')(x)

    # DECODER (smaller)
    x = Conv2DTranspose(16, 3, strides=2, padding='same', activation='relu')(x)
    x = Conv2DTranspose(16, 3, strides=2, padding='same', activation='relu')(x)
    x = Conv2DTranspose(4, 3, strides=2, padding='same', activation='relu')(x)
    # Optional safety resize
    x = Resizing(output_shape[0], output_shape[1], interpolation='bilinear')(x)

    # Final output
    outputs = Conv2D(3, 3, strides=1, padding='same', activation='sigmoid')(x)

    model = Model(inputs, outputs)
    return model


In [ ]:
# SECOND CONFIGURATION
lowq_dir = "/content/frames_lowQ"
highq_dir = "/content/frames_720p"

# Create dataset with downscaling inside the generator
dataset, steps_per_epoch = create_dataset(
    lowq_dir, highq_dir, batch_size=2 # <-- NEW: downscale on the fly
)

# Get input shape from dataset
input_shape = dataset.element_spec[0].shape[1:]  # (H, W, C), now 360x640x3

# Build model
model = build_autoencoder(input_shape, output_shape=(360, 640, 3))

# Compile model
model.compile(optimizer='adam', loss='mse')

# Print summary
model.summary()

# Train
model.fit(dataset, epochs=5, steps_per_epoch=steps_per_epoch)

